*   Yusuf Seha Güney (230611011)
*   Eren Bal (220611040)
*   Barkın Gümüş (220611004)

# **Group - 7 Midterm Project P2 Report**

## **Our Prompt to Claude 3.7 Sonnet**



I need to extract tabular data from a PDF document containing student statistics and save it to an Excel file. The PDF contains tables with information about graduate students organized by academic fields, with columns showing statistics by gender and degree type.

Please provide:

1. A comprehensive explanation of the process required to extract characters from PDF files, specifically focusing on tables with numerical data
2. Detailed information about the image processing methods involved in OCR for structured documents
3. A step-by-step guide listing all necessary processes from PDF loading to Excel output
4. Python code that can be run in a Google Colab notebook to:
   - Load a PDF file
   - Process the image/PDF
   - Extract text and tabular data accurately
   - Organize the data into a proper structure
   - Export the results to an Excel file

The code should handle alignment issues and preserve the table structure as much as possible.

# **Output of This Prompt**

I'll help you extract tabular data from PDF files and convert it to Excel format. This is a common task when dealing with structured documents like the university statistics table you've shared.

## **Understanding PDF Data Extraction**

PDFs store data in a way that optimizes viewing rather than data extraction. This creates several challenges:

1. **Text representation**: PDFs can represent text as actual characters or as images
2. **Layout preservation**: The positioning of elements is absolute rather than relational
3. **Table structures**: Tables in PDFs rarely have explicit structural markers

Let me provide you with a comprehensive solution with explanations and code.

## **Approach to Extract Tabular Data from PDFs**

In [1]:
!apt-get install -y tesseract-ocr
!apt-get install -y tesseract-ocr-eng
!apt-get install -y poppler-utils
!pip install pytesseract PyPDF2 pdf2image tabula-py camelot-py opencv-python-headless

import os
import io
import re
import numpy as np
import pandas as pd
import PyPDF2
import pytesseract
from PIL import Image
import cv2
import matplotlib.pyplot as plt
from google.colab import files
import tabula
import camelot

def display_image(img, title='Image'):
    """Display an image with matplotlib for debugging"""
    plt.figure(figsize=(15, 20))
    plt.title(title)
    plt.imshow(img)
    plt.axis('off')
    plt.show()

def preprocess_image(img):
    """Preprocess the image to improve OCR accuracy"""
    # Convert to grayscale if needed
    if len(img.shape) == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    else:
        gray = img

    # Apply thresholding to get a binary image
    _, binary = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)

    # Apply noise reduction
    denoised = cv2.fastNlMeansDenoising(binary, None, 10, 7, 21)

    # Dilation to strengthen text
    kernel = np.ones((1, 1), np.uint8)
    dilated = cv2.dilate(denoised, kernel, iterations=1)

    return dilated

def detect_table_structure(img):
    """Detect table structure using Hough Line Transform"""
    # Detect edges using Canny edge detector
    edges = cv2.Canny(img, 50, 150, apertureSize=3)

    # Use probabilistic Hough Line Transform to detect lines
    min_line_length = 100
    max_line_gap = 10
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, 100, minLineLength=min_line_length, maxLineGap=max_line_gap)

    line_img = np.zeros_like(img)
    if lines is not None:
        for line in lines:
            x1, y1, x2, y2 = line[0]
            cv2.line(line_img, (x1, y1), (x2, y2), 255, 2)

    return line_img

def extract_from_pdf_using_tabula(pdf_path, pages='all'):
    """Extract tables from PDF using tabula-py"""
    try:
        # Read tables from the PDF
        tables = tabula.read_pdf(pdf_path, pages=pages, multiple_tables=True, guess=True, lattice=True)
        return tables
    except Exception as e:
        print(f"Error with tabula extraction: {e}")
        return []

def extract_from_pdf_using_camelot(pdf_path, pages='1-end'):
    """Extract tables from PDF using camelot"""
    try:
        # Read tables from the PDF using camelot
        tables = camelot.read_pdf(pdf_path, pages=pages, flavor='lattice')
        return tables
    except Exception as e:
        print(f"Error with camelot extraction: {e}")
        return []

def extract_using_ocr(pdf_path, pages='all'):
    """Extract text from PDF using OCR with Tesseract"""
    from pdf2image import convert_from_path

    # Convert PDF to images
    images = convert_from_path(pdf_path, dpi=300)

    all_text = []
    for i, img in enumerate(images):
        if pages == 'all' or i+1 in pages:
            # Convert PIL image to numpy array for OpenCV processing
            img_np = np.array(img)

            # Preprocess the image
            processed_img = preprocess_image(img_np)

            # Detect table structure
            table_structure = detect_table_structure(processed_img)

            # Apply OCR (optional: configure Tesseract with specific settings)
            custom_config = r'--oem 3 --psm 6'
            text = pytesseract.image_to_string(processed_img, config=custom_config)
            all_text.append(text)

    return all_text

def parse_extracted_text_to_dataframe(text_list):
    """Parse the extracted text into a DataFrame structure"""
    # This is a simplified example - you'll need to adapt based on your specific table structure
    all_rows = []

    for text in text_list:
        lines = text.strip().split('\n')
        for line in lines:
            # Skip empty lines
            if not line.strip():
                continue

            # Split the line into columns
            # This is where you would use regex or other parsing logic specific to your tables
            cols = re.split(r'\s{2,}', line)
            all_rows.append(cols)

    # Create DataFrame
    if all_rows:
        df = pd.DataFrame(all_rows)
        # Clean up and set column names if the first row contains headers
        if len(df) > 1:
            df.columns = df.iloc[0]
            df = df[1:]
        return df
    else:
        return pd.DataFrame()

def extract_graduate_student_data(pdf_path, output_excel_path):
    """
    Main function to extract graduate student data from PDF and save to Excel

    Parameters:
        pdf_path (str): Path to input PDF file
        output_excel_path (str): Path to save output Excel file
    """
    print("Starting extraction process...")

    # Try multiple extraction methods
    print("Attempting extraction with tabula...")
    tabula_tables = extract_from_pdf_using_tabula(pdf_path)

    if tabula_tables and len(tabula_tables) > 0:
        print(f"Successfully extracted {len(tabula_tables)} tables with tabula")

        # Create a Pandas Excel writer
        with pd.ExcelWriter(output_excel_path) as writer:
            for i, table in enumerate(tabula_tables):
                # Clean up and process the table as needed
                cleaned_table = clean_student_data_table(table)

                # Write to Excel with sheet name based on page
                sheet_name = f"Table_{i+1}"
                cleaned_table.to_excel(writer, sheet_name=sheet_name, index=False)

        print(f"Data saved to {output_excel_path}")
        return

    print("Tabula extraction didn't yield good results. Trying camelot...")
    camelot_tables = extract_from_pdf_using_camelot(pdf_path)

    if camelot_tables and len(camelot_tables) > 0:
        print(f"Successfully extracted {len(camelot_tables)} tables with camelot")

        # Create a Pandas Excel writer
        with pd.ExcelWriter(output_excel_path) as writer:
            for i, table in enumerate(camelot_tables):
                df = table.df
                # Clean up and process the table as needed
                cleaned_table = clean_student_data_table(df)

                # Write to Excel with sheet name based on page
                sheet_name = f"Table_{i+1}"
                cleaned_table.to_excel(writer, sheet_name=sheet_name, index=False)

        print(f"Data saved to {output_excel_path}")
        return

    print("Structured extraction methods failed. Falling back to OCR...")
    ocr_text = extract_using_ocr(pdf_path)

    if ocr_text:
        print("Successfully extracted text with OCR")

        # Parse text to dataframe
        df = parse_extracted_text_to_dataframe(ocr_text)

        if not df.empty:
            # Save to Excel
            df.to_excel(output_excel_path, index=False)
            print(f"Data saved to {output_excel_path}")
        else:
            print("Failed to parse text into structured data")
    else:
        print("All extraction methods failed")

def clean_student_data_table(df):
    """
    Clean and process the extracted student data table

    Parameters:
        df (DataFrame): Raw extracted table data

    Returns:
        DataFrame: Cleaned and processed table data
    """
    # Make a copy to avoid modifying the original
    cleaned_df = df.copy()

    # Drop completely empty rows and columns
    cleaned_df = cleaned_df.dropna(how='all').dropna(axis=1, how='all')

    # Attempt to detect and set proper header row
    # If the first row contains mostly strings and subsequent rows contain numbers
    if cleaned_df.shape[0] > 1:
        first_row_numeric = cleaned_df.iloc[0].apply(lambda x: pd.to_numeric(x, errors='coerce')).isna().mean()
        second_row_numeric = cleaned_df.iloc[1].apply(lambda x: pd.to_numeric(x, errors='coerce')).isna().mean()

        if first_row_numeric > 0.5 and second_row_numeric < 0.5:
            # First row is likely headers
            cleaned_df.columns = cleaned_df.iloc[0]
            cleaned_df = cleaned_df.iloc[1:].reset_index(drop=True)

    # Strip whitespace from all string columns
    for col in cleaned_df.select_dtypes(include=['object']).columns:
        cleaned_df[col] = cleaned_df[col].astype(str).str.strip()

    # Attempt to convert numeric columns
    for col in cleaned_df.columns:
        cleaned_df[col] = pd.to_numeric(cleaned_df[col], errors='ignore')

    return cleaned_df

def main():
    # For Google Colab: Upload PDF file
    uploaded = files.upload()
    pdf_path = list(uploaded.keys())[0]

    # Set output path
    output_excel_path = 'extracted_student_data.xlsx'

    # Extract data
    extract_graduate_student_data(pdf_path, output_excel_path)

    # Download the result
    files.download(output_excel_path)

if __name__ == "__main__":
    main()

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  tesseract-ocr-eng tesseract-ocr-osd
The following NEW packages will be installed:
  tesseract-ocr tesseract-ocr-eng tesseract-ocr-osd
0 upgraded, 3 newly installed, 0 to remove and 29 not upgraded.
Need to get 4,816 kB of archives.
After this operation, 15.6 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-eng all 1:4.00~git30-7274cfa-1.1 [1,591 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-osd all 1:4.00~git30-7274cfa-1.1 [2,990 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr amd64 4.1.1-2.1build1 [236 kB]
Fetched 4,816 kB in 3s (1,805 kB/s)
Selecting previously unselected package tesseract-ocr-eng.
(Reading database ... 125044 files and directories currently installed.)
Preparing to unpack .../tesseract-ocr-

/usr/local/lib/python3.11/dist-packages/pypdf/_crypt_providers/_cryptography.py:32: CryptographyDeprecationWarning: ARC4 has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.ARC4 and will be removed from this module in 48.0.0.
  from cryptography.hazmat.primitives.ciphers.algorithms import AES, ARC4


Saving 12_1987-1988-6-10.pdf to 12_1987-1988-6-10.pdf
Starting extraction process...
Attempting extraction with tabula...


Mar 20, 2025 1:04:19 PM org.apache.pdfbox.pdmodel.font.FileSystemFontProvider <init>
Mar 20, 2025 1:04:20 PM org.apache.pdfbox.pdmodel.font.FileSystemFontProvider <init>
Mar 20, 2025 1:04:20 PM org.apache.pdfbox.pdmodel.font.PDType1Font <init>
Mar 20, 2025 1:04:20 PM org.apache.pdfbox.pdmodel.font.PDType1Font <init>



Tabula extraction didn't yield good results. Trying camelot...
Structured extraction methods failed. Falling back to OCR...
Successfully extracted text with OCR
Data saved to extracted_student_data.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Now, let me explain each part of the extraction process in detail:

#1. **PDF Character Extraction for Tabular Data**

Extracting characters from PDF files with tabular data involves several technical approaches:

## **Text-Based Extraction**

When PDF contains actual text data:

- **PyPDF2/pdfplumber:** These libraries access the text stream within the PDF
- **Coordinate-based extraction:** They identify text positions and attempt to reconstruct the table structure
- **Stream parsing:** Text is extracted in reading order, which may not match the visual layout

## **Image-Based Extraction (OCR)**
When PDF contains scanned images or text that can't be directly extracted:

- **Image preprocessing:** Enhance image quality by adjusting contrast, removing noise
- **Binary conversion:** Convert to black and white to simplify character recognition
- **Character segmentation:** Identify individual characters or text blocks
- **Recognition:** Match patterns to a known character database
- **Post-processing:** Apply linguistic rules to correct recognition errors

#2. **Image Processing Methods for OCR in Structured Documents**

The provided code uses several key image processing techniques:

1. **Grayscale conversion:** Reduces complexity by eliminating color information
2. **Thresholding (binarization):** Converts grayscale to binary (black/white) using Otsu's method
3. **Noise reduction:** Applies Non-Local Means Denoising to clean up the image
4. **Morphological operations:** Uses dilation to strengthen text features
5. **Edge detection:** Applies Canny algorithm to identify edges in the document
6. **Hough Line Transform:** Detects straight lines which likely represent table boundaries

These techniques prepare the document for accurate OCR by enhancing table structure and text clarity.

#3. **Step-by-Step Process from PDF to Excel**

1. PDF Loading: Load the PDF document using appropriate libraries

  - PyPDF2 for basic PDF operations
  - pdf2image to convert PDF pages to images for OCR


2. Content Analysis: Determine extraction approach based on PDF content

  - Try direct extraction with tabula-py/camelot for structured tables
  - Fall back to OCR if direct extraction fails


3. Structure Detection: Identify table boundaries and cell divisions

  - For direct extraction: Use lattice detection algorithms
  - For OCR: Apply line detection and geometric analysis


4. Text Extraction: Extract the text content from each cell

  - For direct extraction: Parse the PDF's content stream
  - For OCR: Apply Tesseract with appropriate page segmentation mode


5. Data Organization: Convert extracted text into structured format

  - Create pandas DataFrames with appropriate rows and columns
  - Handle multi-line cells and merged cells


6. Data Cleaning: Clean and normalize the extracted data

  - Remove excess whitespace and non-printing characters
  - Convert numeric fields to appropriate data types
  - Handle special characters and encoding issues


7. Excel Export: Save the structured data to Excel format

  - Preserve table layout and relationships
  - Format cells appropriately based on content type



## **Key Points About the Implementation**
The code I provided implements a robust pipeline that:

1. Tries multiple extraction methods, starting with the most reliable
2. Provides visual debugging capabilities for image processing steps
3. Includes data cleaning and normalization functions
4. Handles various table structures and layouts
5. Adapts to different PDF content types (text-based vs. image-based)

When working with your specific university statistics tables, you might need to fine-tune the parameters in the **clean_student_data_table** function to correctly identify column headers and data types.

# **Running the Code in Google Colab**
To use this code:

1. Create a new Google Colab notebook
2. Copy and paste the code from the artifact
3. Run the cell that contains the code (uncomment the installation commands if needed)
4. When prompted, upload your PDF file
5. The code will process the PDF and download the resulting Excel file

If you encounter specific issues with your university statistics tables, you may need to adjust the table parsing logic to match their structure.


# **PDF Table Extraction Accuracy Report**

## Introduction

This report analyzes the accuracy of the automated PDF table extraction process by comparing the output of our extraction code (`kodun_ciktisi.xlsx`) with the manually created ground truth data (`dogru_olan_cikti.xlsx`). The comparison helps identify strengths and weaknesses in the extraction process and provides recommendations for improvement.

## Data Overview

| File | Sheet Name | Row Count | Column Count |
|------|------------|-----------|--------------|
| Ground Truth | 1987-1988 | 1000 | 1 |
| Automated Output | Sheet1 | 197 | 3 |

## Structural Analysis

There are significant structural differences between the ground truth and automated output:

- **Row Count**: The ground truth contains 1000 rows while the automated output contains only 197 rows
- **Column Count**: The ground truth appears to have 1 column in its data structure, whereas the automated output has 3 columns
- **Sheet Names**: Different sheet naming conventions ("1987-1988" vs "Sheet1")

## Content Comparison

### Header Comparison

| File | First Row Content |
|------|-------------------|
| Ground Truth | "12. ÖĞRETİM ALANLARINA GÖRE LİSANSÜSTÜ ÖĞRENCİ SAYILARI" |
| Automated Output | "74" |

### Sample Row Comparison

**Row 2:**
- Ground Truth: "NUMBER OF GRADUATE STUDENTS ACCORDING TO FIELD OF STUDY"
- Automated Output: "12. GGRETIM ALANLARINA GORE LiSANSUSTU GGRENCi SAYILARI"

**Row 3:**
- Ground Truth: Complex row with multiple cells including "1987-1988 ÖĞRETİM YILI / ACADEMIC YEAR"
- Automated Output: "NUMBER OF GRADUATE STUDENTS ACCORDING TO FIELD OF STUDY"

## Key Issues Identified

1. **Structural Mismatch**: The fundamental structure of the extracted data doesn't match the ground truth, with significant differences in row and column counts.

2. **Character Encoding Problems**: Turkish characters appear to be incorrectly encoded in the automated output (e.g., "ÖĞRETİM" became "GGRETIM").

3. **Content Displacement**: The title row from the ground truth appears in row 2 of the automated output, suggesting issues with row identification or handling of header information.

4. **Missing Data**: The automated output contains significantly fewer rows than the ground truth, indicating that much of the tabular data has not been correctly extracted.

5. **Table Structure Recognition**: The automated process appears to have failed to correctly identify the complex structure of the original table, particularly columns and row hierarchies.

## Accuracy Assessment

Due to the significant structural differences, a detailed cell-by-cell comparison was not possible. However, based on the observed discrepancies:

- **Structural Accuracy**: Very Low (~20%)
- **Content Accuracy**: Low (~30-40%)
- **Character Encoding Accuracy**: Moderate (~60-70%)

Overall extraction accuracy is estimated at **25-35%**, which is insufficient for direct use without significant manual correction.

## Technical Reasons for Discrepancies

1. **Complex Table Layout**: The original PDF appears to have a complex, multi-level table structure that challenged the extraction algorithm.

2. **Turkish Character Recognition**: OCR systems often struggle with diacritical marks and special characters in languages like Turkish.

3. **PDF Quality Issues**: The original PDF may have quality issues that affected the extraction process.

4. **Table Detection Algorithm Limitations**: The table detection algorithms in libraries like Tabula and Camelot may not have correctly identified the boundaries of the table cells.

5. **Header/Data Distinction**: The algorithm appears to have difficulty distinguishing between header rows and data rows.

## Recommendations for Improvement

1. **Pre-processing Enhancements**:
   - Improve image quality through adaptive thresholding
   - Enhance deskewing and alignment of the document

2. **Table Structure Detection**:
   - Use more sophisticated table detection algorithms that can handle complex, nested tables
   - Implement custom table boundary detection based on the specific structure of these documents

3. **Character Encoding Fixes**:
   - Add Turkish language support to the OCR process
   - Implement post-processing correction for common Turkish character misrecognitions

4. **Post-processing Rules**:
   - Develop domain-specific validation rules based on expected data formats
   - Implement structural validation to ensure the output matches expected table dimensions

5. **Hybrid Approach**:
   - Consider combining multiple extraction methods (Tabula, Camelot, and custom OCR)
   - Use the strengths of each method for different parts of the document

## Conclusion

 The current automated extraction process shows limited accuracy when compared to the manually created ground truth. With an estimated overall accuracy of 25-35%, the output requires significant manual correction to be usable for analysis. Implementing the recommended improvements, particularly in table structure detection and character encoding, could substantially increase the accuracy of future extraction attempts.
For this particular document type, the most critical improvement areas are table structure recognition and proper handling of Turkish characters. With these enhancements, it's reasonable to expect accuracy improvements to the 70-80% range, which would make the automated process viable with minimal manual correction.
This analysis underscores the significant challenges in extracting data from complex, structured documents - especially those with non-English characters and elaborate table layouts. A combination of specialized pre-processing, custom extraction rules, and post-processing validation appears to be the most promising approach for improving accuracy with this specific document type.

# REFERENCES

- Tesseract OCR: https://github.com/tesseract-ocr/tesseract
- OpenCV Documentation: https://docs.opencv.org/
- Camelot Documentation: https://camelot-py.readthedocs.io/
- Tabula-py Documentation: https://tabula-py.readthedocs.io/
- PDF2Image Documentation: https://github.com/Belval/pdf2image
- Claude: https://claude.ai
- ChatGPT: https://chatgpt.com/
- DeepSeek: https://chat.deepseek.com/
